# InferStack: Kaggle T4 x2 FP16 serving smoke

Choose Settings > Accelerator > GPU T4 x2, enable Internet, and add a read-only GITHUB_TOKEN secret with notebook access. Run cells in order. This is a small-model vLLM check on GPU 0 and GPU 1 separately, not a BF16/native qualification and not one model split over two GPUs. Uses a pinned Qwen3-0.6B revision and a dedicated vLLM 0.11.0 environment; it does not pair that old engine with Qwen3.5.

Prepared from source and syntax-checked locally; the exact notebook environment has not yet been GPU-qualified. Setup or conformance failures are evidence to retain, not gates to disable. No optimizer API credentials are needed. Download the final evidence ZIP; Kaggle session storage is temporary.


## 1. Verify GPU, clone privately, and pin source


In [ ]:
import base64, datetime, getpass, json, os, signal, subprocess, sys, time
from pathlib import Path
PLATFORM = 'kaggle'
COMMIT = 'c5db2f1c16d9f10fd057c14ab76edc89efe7d0e0'
BASE = Path('/content' if PLATFORM == 'colab' else '/kaggle/working')
REPO = BASE / 'inferstack'
assert (3, 11) <= sys.version_info[:2] < (3, 14), 'This profile needs Python 3.11–3.13.'
gpu = subprocess.check_output(['nvidia-smi', '--query-gpu=name,memory.total', '--format=csv,noheader'], text=True)
print(gpu)
if PLATFORM == 'colab':
    assert 'A100' in gpu, 'Choose an A100 runtime for this notebook. Use the separate T4 notebook for Kaggle.'
else:
    assert len([line for line in gpu.splitlines() if 'T4' in line]) >= 2, 'Choose Kaggle GPU T4 x2.'

def clone_pinned():
    if not (REPO / '.git').is_dir():
        if PLATFORM == 'kaggle':
            from kaggle_secrets import UserSecretsClient
            token = UserSecretsClient().get_secret('GITHUB_TOKEN').strip()
        else:
            token = getpass.getpass('Read-only GitHub token for this private repo: ').strip()
        env = os.environ.copy()
        env.update(GIT_TERMINAL_PROMPT='0', GIT_CONFIG_COUNT='1',
                   GIT_CONFIG_KEY_0='http.https://github.com/.extraheader',
                   GIT_CONFIG_VALUE_0='AUTHORIZATION: basic '+base64.b64encode(('x-access-token:'+token).encode()).decode())
        result = subprocess.run(['git','clone','-q','https://github.com/sanjoxtech/inferstack.git',str(REPO)],
                                env=env,capture_output=True,text=True)
        del token, env
        if result.returncode:
            raise RuntimeError('Clone failed. Check read-only repository access and Internet connectivity. Credentials were not printed or stored.')
    subprocess.run(['git','checkout','--detach',COMMIT],cwd=REPO,check=True,capture_output=True,text=True)
    actual=subprocess.check_output(['git','rev-parse','HEAD'],cwd=REPO,text=True).strip()
    assert actual == COMMIT
    print('Pinned source:', actual)
clone_pinned()
RUN_ID=datetime.datetime.now(datetime.timezone.utc).strftime('%Y%m%dT%H%M%SZ')
OUT=REPO/'results'/f'{PLATFORM}-h0-{COMMIT[:7]}-{RUN_ID}'
OUT.mkdir(parents=True,exist_ok=False)
(OUT/'gpu-inventory.txt').write_text(gpu)
(OUT/'run-context.json').write_text(json.dumps({'commit':COMMIT,'platform':PLATFORM,'run_id':RUN_ID},indent=2))

def logged(argv, filename, *, env=None, timeout=3600):
    path=OUT/filename
    start=time.monotonic()
    with path.open('w') as log:
        proc=subprocess.Popen(list(map(str,argv)),cwd=REPO,env=env,stdout=log,stderr=subprocess.STDOUT,start_new_session=True)
        try:
            while proc.poll() is None:
                elapsed=time.monotonic()-start
                if elapsed > timeout:
                    raise TimeoutError(f'Wall-time cap reached: {filename}')
                print(f'{filename}: running, {int(elapsed)} seconds elapsed',flush=True)
                time.sleep(20)
        except BaseException:
            try:
                os.killpg(proc.pid,signal.SIGTERM)
                proc.wait(timeout=20)
            except (ProcessLookupError,subprocess.TimeoutExpired):
                try: os.killpg(proc.pid,signal.SIGKILL)
                except ProcessLookupError: pass
            raise
    print(filename,'exit:',proc.returncode)
    if proc.returncode:
        print(path.read_text(errors='replace')[-3000:])
    return proc.returncode


## 2. Install and record the isolated environment


In [ ]:
# Separate environment: vLLM chooses its matching torch/Transformers dependencies.
# Do not install InferStack's native/HF extras into this environment.
VENV=Path('/tmp/inferstack-vllm-env')
subprocess.run([sys.executable,'-m','venv',str(VENV)],check=True)
PY=VENV/'bin/python'
ENV={**os.environ,'PATH':str(VENV/'bin')+os.pathsep+os.environ['PATH'],
     'PYTHONPATH':str(REPO/'src'),'INFERSTACK_PROVIDER':'kaggle','HF_HOME':'/tmp/inferstack-kaggle-hf'}
assert logged([PY,'-m','pip','install','--no-cache-dir','-e','.[dev,postgres]','vllm==0.11.0'],
              'install-vllm.log',env=ENV,timeout=2700)==0
assert logged([PY,'-c',"import torch,vllm; assert torch.cuda.is_available(); assert torch.cuda.device_count()>=2; print('torch',torch.__version__,'CUDA',torch.version.cuda,'vLLM',vllm.__version__); print([torch.cuda.get_device_name(i) for i in range(torch.cuda.device_count())])"],
              'preflight.log',env=ENV,timeout=180)==0
print((OUT/'preflight.log').read_text())
(OUT/'pip-freeze.txt').write_text(subprocess.check_output([str(PY),'-m','pip','freeze'],env=ENV,text=True))
MODEL='Qwen/Qwen3-0.6B'
MODEL_REVISION='c1899de289a04d12100db370d81485cdf75e47ca'
(OUT/'model-identity.json').write_text(json.dumps({'model':MODEL,'revision':MODEL_REVISION,'dtype':'float16'},indent=2))


## 3. Check each GPU independently


In [ ]:
# The existing conformance runner is invoked directly so launch extras remain a list.
# Each check gets one identified GPU. This is not tensor parallelism or a speedup claim.
for device in (0,1):
    target=OUT/f'conformance-t4-gpu{device}.json'
    args={'engine':'vllm','model':MODEL,'revision':MODEL_REVISION,'dtype':'float16',
          'ready_timeout':1200,'out':str(target.relative_to(REPO)),
          'extra':['--max-model-len','2048','--max-num-seqs','8',
                   '--gpu-memory-utilization','0.60','--enforce-eager']}
    snippet=("import asyncio,runpy; from types import SimpleNamespace; "
             "runner=runpy.run_path('scripts/engine_conformance.py'); "
             f"raise SystemExit(asyncio.run(runner['run'](SimpleNamespace(**{args!r}))))")
    env={**ENV,'CUDA_VISIBLE_DEVICES':str(device)}
    code=logged([PY,'-u','-c',snippet],f'conformance-t4-gpu{device}.runner.log',env=env,timeout=1800)
    print('Physical GPU',device,'runner exit',code)


## 4. Read the verdict


In [ ]:
summary=[]
for device in (0,1):
    path=OUT/f'conformance-t4-gpu{device}.json'
    result=json.loads(path.read_text()) if path.exists() else {'pass':False,'error':'no conformance report'}
    summary.append({'physical_gpu':device,'pass':result.get('pass') is True,'checks':result.get('checks'),
                    'error':result.get('error'),'model':MODEL,'revision':MODEL_REVISION,'dtype':'float16'})
    print(json.dumps(summary[-1],indent=2))
(OUT/'t4-serving-summary.json').write_text(json.dumps({'scope':'two independent FP16 serving replicas',
    'full_v2_qualified':False,'replicas':summary},indent=2))
print('This is T4/vLLM/0.6B serving evidence only. It does not qualify native BF16, A100, multi-node serving or the full v2 release.')


## 5. Download evidence


In [ ]:
import shutil
archive_base=BASE/f'inferstack-{PLATFORM}-evidence-{RUN_ID}'
zip_path=shutil.make_archive(str(archive_base),'zip',root_dir=OUT.parent,base_dir=OUT.name)
print('Evidence archive:',zip_path)
if PLATFORM=='colab':
    from google.colab import files
    files.download(zip_path)
else:
    from IPython.display import FileLink, display
    display(FileLink(zip_path))
    print('You can also download the ZIP from the notebook file browser under /kaggle/working.')
